# Vision Computacional — MyCobot 280
### Rol 4: Ingeniero de Vision — Aaron
**Problema que cubre:** P6 (Deteccion de Objetos y Transformacion pixel a mm)

---
**Orden de ejecucion:**
1. Celda 1 — Conexion a la camara
2. Celda 2 — Calibracion HSV
3. Celda 3 — Deteccion de contorno y centroide
4. Celda 4 — Transformacion pixel a mm
5. Celda 5 — Funcion principal detect_object
6. Celda 6 — Validacion en 3 posiciones
7. Celda 7 — Manejo de casos especiales

In [ ]:
# ============================================================
# CELDA 1 — CONEXION A LA CAMARA
# Inicializa la camara y verifica que este funcionando
# Indice 0 = camara principal del Jetson Nano
# ============================================================

import cv2
import numpy as np
import time
import ipywidgets.widgets as widgets
from IPython.display import display

# Abrir camara
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Verificar conexion
if cap.isOpened():
    print("Camara conectada correctamente")
    print(f"Resolucion: {int(cap.get(3))} x {int(cap.get(4))} pixeles")
else:
    print("ERROR: No se pudo conectar la camara")

# Capturar un frame de prueba
ret, frame = cap.read()
if ret:
    print("Frame capturado correctamente")
    print(f"Dimensiones del frame: {frame.shape}")
else:
    print("ERROR: No se pudo capturar frame")

In [ ]:
# ============================================================
# CELDA 2 — CALIBRACION HSV
# Define los rangos de color HSV para detectar el objeto
# HSV = Hue (tono), Saturation (saturacion), Value (brillo)
#
# Rangos de referencia para colores comunes:
#   Rojo:    H=0-10  o H=160-180, S=100-255, V=100-255
#   Verde:   H=40-80,  S=50-255,  V=50-255
#   Azul:    H=100-130, S=50-255, V=50-255
#   Amarillo: H=20-35, S=100-255, V=100-255
#
# INSTRUCCION: Ajusta los valores segun el color de tu objeto
# en las condiciones de iluminacion del laboratorio
# ============================================================

# Rango HSV calibrado para el objeto en el laboratorio
# Ajustar estos valores segun el color real del objeto
HSV_BAJO  = np.array([0,   120, 70])   # Limite inferior HSV
HSV_ALTO  = np.array([10,  255, 255])  # Limite superior HSV

# Para objetos rojos (el rojo en HSV aparece en dos rangos)
HSV_BAJO2 = np.array([160, 120, 70])   # Segundo rango para rojo
HSV_ALTO2 = np.array([180, 255, 255])

# Capturar frame y mostrar la mascara de color
ret, frame = cap.read()
if ret:
    # Convertir de BGR (OpenCV) a HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Crear mascara con el rango de color definido
    mask1 = cv2.inRange(hsv, HSV_BAJO,  HSV_ALTO)
    mask2 = cv2.inRange(hsv, HSV_BAJO2, HSV_ALTO2)
    mask  = cv2.bitwise_or(mask1, mask2)  # Combinar ambas mascaras

    # Reducir ruido con operaciones morfologicas
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)  # Eliminar puntos pequenos
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # Rellenar huecos

    # Guardar frame y mascara para verificacion visual
    cv2.imwrite('/tmp/frame_original.jpg', frame)
    cv2.imwrite('/tmp/mascara_color.jpg',  mask)

    print("Calibracion HSV aplicada")
    print(f"HSV bajo:  {HSV_BAJO}")
    print(f"HSV alto:  {HSV_ALTO}")
    print("Imagenes guardadas en /tmp/ para verificacion")
else:
    print("ERROR: No se pudo capturar frame para calibracion")

In [ ]:
# ============================================================
# CELDA 3 — DETECCION DE CONTORNO Y CENTROIDE
# Detecta el contorno del objeto en la mascara y calcula
# el centroide (cx, cy) en pixeles usando momentos de imagen
# ============================================================

# Area minima del contorno para filtrar falsos positivos
# Ajustar segun el tamano del objeto en la imagen
AREA_MINIMA = 500  # pixeles cuadrados

def detectar_centroide(frame):
    """
    Detecta el centroide del objeto en el frame.
    Retorna (cx, cy) en pixeles o None si no hay objeto.

    Proceso:
    1. Convertir frame BGR a HSV
    2. Aplicar mascara de color HSV
    3. Reducir ruido con morfologia
    4. Encontrar contornos
    5. Seleccionar el contorno mas grande
    6. Calcular centroide con momentos
    """
    # Convertir a HSV y crear mascara
    hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask1 = cv2.inRange(hsv, HSV_BAJO,  HSV_ALTO)
    mask2 = cv2.inRange(hsv, HSV_BAJO2, HSV_ALTO2)
    mask  = cv2.bitwise_or(mask1, mask2)

    # Reducir ruido
    kernel = np.ones((5, 5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Encontrar contornos en la mascara
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Verificar que hay contornos detectados
    if not contours:
        return None  # No se detecto ningun objeto

    # Seleccionar el contorno de mayor area
    contorno_mayor = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contorno_mayor)

    # Filtrar contornos pequenos (falsos positivos)
    if area < AREA_MINIMA:
        return None  # Objeto demasiado pequeno, ignorar

    # Calcular centroide usando momentos de imagen
    M = cv2.moments(contorno_mayor)
    if M['m00'] == 0:
        return None  # Evitar division por cero

    cx = int(M['m10'] / M['m00'])  # Coordenada x del centroide
    cy = int(M['m01'] / M['m00'])  # Coordenada y del centroide

    return (cx, cy)

# Probar la deteccion con el frame actual
ret, frame = cap.read()
if ret:
    resultado = detectar_centroide(frame)
    if resultado:
        cx, cy = resultado
        print(f"Objeto detectado en: cx={cx} px, cy={cy} px")
    else:
        print("Objeto no detectado — ajusta los rangos HSV")
else:
    print("ERROR: No se pudo capturar frame")

In [ ]:
# ============================================================
# CELDA 4 — TRANSFORMACION PIXEL A MM
# Convierte coordenadas (cx, cy) en pixeles a coordenadas
# (x_mm, y_mm) en el sistema de referencia del robot
#
# Metodo: transformacion lineal basada en puntos de referencia
# medidos fisicamente en el laboratorio
#
# INSTRUCCION: Para calibrar, coloca el objeto en posiciones
# conocidas y mide tanto los pixeles (camara) como los mm
# (con mc.get_coords()) para calcular los factores de escala
# ============================================================

# Parametros de calibracion camara-robot
# Estos valores se obtienen midiendo puntos reales en el laboratorio

# Centro de la imagen en pixeles (referencia)
CX_CENTRO = 320  # pixeles (mitad del ancho 640px)
CY_CENTRO = 240  # pixeles (mitad del alto 480px)

# Factores de conversion pixel a mm
# Calibrar midiendo: desplazamiento real (mm) / desplazamiento en pixeles
ESCALA_X = 0.85  # mm por pixel en eje X (ajustar con calibracion real)
ESCALA_Y = 0.85  # mm por pixel en eje Y (ajustar con calibracion real)

# Offset: posicion del robot cuando apunta al centro de la imagen
# Obtener con mc.get_coords() cuando el robot apunta al centro de la mesa
OFFSET_X =  50.0  # mm (ajustar con calibracion real)
OFFSET_Y = -65.0  # mm (ajustar con calibracion real)

def pixel_a_mm(cx, cy):
    """
    Convierte coordenadas de pixeles a milimetros
    en el sistema de referencia del robot.

    Formula:
        x_mm = OFFSET_X + (cx - CX_CENTRO) * ESCALA_X
        y_mm = OFFSET_Y - (cy - CY_CENTRO) * ESCALA_Y

    Nota: el eje Y se invierte porque en imagen Y crece
    hacia abajo, pero en el robot Y crece hacia arriba.
    """
    x_mm = OFFSET_X + (cx - CX_CENTRO) * ESCALA_X
    y_mm = OFFSET_Y - (cy - CY_CENTRO) * ESCALA_Y
    return round(x_mm, 1), round(y_mm, 1)

# Probar la transformacion con el centroide detectado
ret, frame = cap.read()
if ret:
    resultado = detectar_centroide(frame)
    if resultado:
        cx, cy = resultado
        x_mm, y_mm = pixel_a_mm(cx, cy)
        print(f"Centroide en pixeles: cx={cx}, cy={cy}")
        print(f"Posicion en mm:       x={x_mm}mm, y={y_mm}mm")
    else:
        print("Objeto no detectado")

In [ ]:
# ============================================================
# CELDA 5 — FUNCION PRINCIPAL detect_object
# Funcion requerida por el Lider de Integracion (Leon)
# para ser consumida en main.py
#
# Entrega: detect_object(frame) -> (x_mm, y_mm) | None
# ============================================================

def detect_object(frame):
    """
    Detecta el objeto objetivo en el frame y retorna
    su posicion en el sistema de referencia del robot.

    Parametros:
        frame: imagen BGR capturada con cv2.VideoCapture

    Retorna:
        (x_mm, y_mm): posicion del objeto en milimetros
        None: si no se detecta ningun objeto valido

    Manejo de casos especiales:
        - Frame vacio o None: retorna None
        - Objeto no encontrado: retorna None
        - Multiples objetos: toma el de mayor area
        - Falso positivo (area pequena): retorna None
    """
    # Validar que el frame sea valido
    if frame is None or frame.size == 0:
        print("ADVERTENCIA: Frame vacio o invalido")
        return None

    # Detectar centroide en pixeles
    centroide = detectar_centroide(frame)

    # Verificar que se detecto un objeto
    if centroide is None:
        return None  # Objeto no encontrado en la escena

    # Convertir de pixeles a mm en sistema de referencia del robot
    cx, cy = centroide
    x_mm, y_mm = pixel_a_mm(cx, cy)

    return (x_mm, y_mm)

# Probar la funcion principal con el frame actual
ret, frame = cap.read()
if ret:
    resultado = detect_object(frame)
    if resultado:
        x_mm, y_mm = resultado
        print(f"Objeto detectado en: x={x_mm}mm, y={y_mm}mm")
        print("Listo para enviar al Lider de Integracion")
    else:
        print("Objeto no detectado en la escena actual")
else:
    print("ERROR: No se pudo capturar frame")

In [ ]:
# ============================================================
# CELDA 6 — VALIDACION EN 3 POSICIONES
# Compara la posicion detectada vs la posicion real medida
# con mc.get_coords() para calcular el error en mm
#
# INSTRUCCION: Coloca el objeto en 3 posiciones distintas,
# mide con get_coords() la posicion real del robot sobre el
# objeto, y corre esta celda para cada posicion
# ============================================================

# Posiciones reales medidas con mc.get_coords() en el laboratorio
# Formato: (x_real_mm, y_real_mm)
# SUSTITUIR con los valores reales medidos
POSICIONES_REALES = [
    (50.0,  -65.0),   # Posicion 1: centro de la mesa
    (150.0, -65.0),   # Posicion 2: desplazado a la derecha
    (50.0,  -165.0),  # Posicion 3: desplazado al frente
]

def validar_deteccion():
    """
    Captura un frame, detecta el objeto y compara
    con la posicion real. Registra el error en mm.
    """
    print("=" * 55)
    print("TABLA DE VALIDACION — Error de deteccion (mm)")
    print("=" * 55)
    print(f"{'Pos':>4} {'Real X':>8} {'Real Y':>8} {'Det X':>8} {'Det Y':>8} {'Error X':>8} {'Error Y':>8}")
    print("-" * 55)

    for i, (x_real, y_real) in enumerate(POSICIONES_REALES):
        # Capturar frame en la posicion actual
        ret, frame = cap.read()
        if not ret:
            print(f"  {i+1}  ERROR al capturar frame")
            continue

        # Detectar posicion del objeto
        resultado = detect_object(frame)

        if resultado:
            x_det, y_det = resultado
            error_x = round(abs(x_real - x_det), 1)
            error_y = round(abs(y_real - y_det), 1)
            print(f"  {i+1:>2}  {x_real:>8.1f} {y_real:>8.1f} {x_det:>8.1f} {y_det:>8.1f} {error_x:>8.1f} {error_y:>8.1f}")
        else:
            print(f"  {i+1:>2}  {x_real:>8.1f} {y_real:>8.1f}  No detectado")

        time.sleep(1)  # Pausa entre capturas

    print("-" * 55)
    print("Error aceptable segun rubrica: < 15mm")

# INSTRUCCION: Coloca el objeto en la Posicion 1 y corre la celda
# Luego repite para las posiciones 2 y 3
validar_deteccion()

In [ ]:
# ============================================================
# CELDA 7 — MANEJO DE CASOS ESPECIALES
# Prueba el comportamiento del sistema ante:
#   1. Ausencia del objeto en la escena
#   2. Falsos positivos por iluminacion variable
#   3. Deteccion continua en tiempo real
# ============================================================

def detectar_con_reintento(intentos=3, espera=0.5):
    """
    Intenta detectar el objeto varias veces antes de reportar
    que no esta en la escena. Reduce falsos negativos por
    variaciones de iluminacion momentaneas.

    Parametros:
        intentos: numero de capturas antes de reportar fallo
        espera:   segundos entre cada intento
    """
    for i in range(intentos):
        ret, frame = cap.read()
        if not ret:
            continue

        resultado = detect_object(frame)
        if resultado:
            return resultado  # Objeto encontrado

        time.sleep(espera)  # Esperar antes del siguiente intento

    return None  # Objeto no encontrado despues de todos los intentos

def deteccion_continua(duracion_seg=5):
    """
    Detecta el objeto continuamente durante un tiempo definido.
    Util para verificar la estabilidad de la deteccion.
    """
    print(f"Deteccion continua durante {duracion_seg} segundos...")
    inicio = time.time()
    detecciones = 0
    frames_total = 0

    while (time.time() - inicio) < duracion_seg:
        ret, frame = cap.read()
        if not ret:
            continue

        frames_total += 1
        resultado = detect_object(frame)

        if resultado:
            x_mm, y_mm = resultado
            detecciones += 1
            print(f"Frame {frames_total:>3}: objeto en x={x_mm:>7.1f}mm, y={y_mm:>7.1f}mm")
        else:
            print(f"Frame {frames_total:>3}: objeto no detectado")

        time.sleep(0.3)  # ~3 frames por segundo

    # Resumen de estabilidad
    tasa = (detecciones / frames_total * 100) if frames_total > 0 else 0
    print("\n" + "=" * 40)
    print("RESUMEN DE ESTABILIDAD")
    print("=" * 40)
    print(f"Frames totales:    {frames_total}")
    print(f"Detecciones OK:    {detecciones}")
    print(f"Tasa de deteccion: {tasa:.1f}%")

# PRUEBA 1: Objeto ausente (quitar el objeto y correr)
print("=" * 40)
print("PRUEBA 1: Deteccion con reintento")
print("=" * 40)
resultado = detectar_con_reintento(intentos=3)
if resultado:
    print(f"Objeto detectado en: {resultado}")
else:
    print("Objeto no encontrado en la escena")

# PRUEBA 2: Deteccion continua para verificar estabilidad
print("\n" + "=" * 40)
print("PRUEBA 2: Estabilidad de deteccion (5s)")
print("=" * 40)
deteccion_continua(duracion_seg=5)

In [ ]:
# ============================================================
# CELDA 8 — CIERRE DE CAMARA
# Siempre correr esta celda al terminar para liberar
# el recurso de la camara y evitar conflictos con otros
# notebooks que necesiten /dev/video0
# ============================================================

cap.release()
print("Camara liberada correctamente")
print("detect_object lista para ser importada en main.py")